### 01 – Data Preparation

Runs once, on CPU.

**Scope:** load and inspect the raw data, clean it, split it into `train` / `val` / `test`, and save the result under `data/processed/`. Once the split is saved it is **frozen** — all later experiments use exactly the same data.

**Rationale:** all data-processing logic lives in `src/data.py`. This notebook only calls those functions, displays results, and saves files, so that a change to a threshold or a cleaning rule cannot silently alter the dataset in a way that makes experiments incomparable.

## 0 - Bootstrap

Two jobs: make `src/` importable and record the environment.

Version tracking matters for reproducibility: scikit-learn versions can affect `train_test_split`, and `transformers < 4.51` does not support Qwen3. Both versions are saved in the manifest so any differences can be explained.

In [1]:
import sys, json, platform
from pathlib import Path

# Works both locally and in Colab: find the repo root, then put it on sys.path
# so that `import src.data` resolves the same way in both places.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import sklearn, scipy

import src.data as D

LIBRARY_VERSIONS = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit-learn": sklearn.__version__,
    "scipy": scipy.__version__,
}
# Only the folder name, never the absolute path: the path contains the local
# user name and (on this machine) non-English directory names, and neither
# belongs in a notebook that gets committed and submitted.
print("repo root :", REPO_ROOT.name, "| src importable:", (REPO_ROOT / "src" / "data.py").exists())
for k, v in LIBRARY_VERSIONS.items():
    print(f"{k:>13} : {v}")
print("\nsplit seed:", D.SPLIT_SEED,
      "| near-duplicate threshold:", D.NEAR_DUP_THRESHOLD, "(derived in 1.3)")
# Tables that the report needs must land on disk, not only in a cell output.
# A number that exists only as notebook output gets retyped by hand into the
# report from a screenshot, and that is how a wrong figure gets published.
METRICS_DIR = REPO_ROOT / "results" / "metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)


repo root : support-triage | src importable: True
       python : 3.13.14
       pandas : 3.0.3
        numpy : 2.5.1
 scikit-learn : 1.9.0
        scipy : 1.18.0

split seed: 42 | near-duplicate threshold: 0.9 (derived in 1.3)


### 1.1 - Load the file and verify it by hand

Before anything else, verify **26,872 rows, 27 intents, and 11 categories**. If any number changes, the upstream dataset was updated, so `verify_raw` raises an error rather than a warning.



In [2]:
raw_path = REPO_ROOT / "data" / "raw" / D.RAW_FILENAME
df_raw = D.load_raw(REPO_ROOT / "data" / "raw")
df_raw = df_raw.reset_index(names="row_id")   # keep a stable pointer back to the raw file

print("verified:", D.verify_raw(df_raw))
print("sha256 of the raw file:", D.sha256_of_file(raw_path)[:16], "...")
print("\nshape:", df_raw.shape)
df_raw.head(3)

verified: {'n_rows': 26872, 'n_intents': 27, 'n_categories': 11}
sha256 of the raw file: 6f81102b0100b97b ...

shape: (26872, 6)


,row_id,flags,instruction,category,intent,response
0,0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...


### What each column is for, and the two traps

| column        | role                                                                          |
| ------------- | ----------------------------------------------------------------------------- |
| `instruction` | Customer message — **the only model input**                                   |
| `intent`      | Label — what the model predicts                                               |
| `category`    | Not trained; derived from `intent`                                            |
| `flags`       | Dataset-generation metadata — **not a feature**; useful for error analysis    |
| `response`    | Templated answer — **excluded from training and not used as a model feature** |

**Trap 1 — `response` leakage:** Using everything except `intent` as features would expose the answer to the model, producing artificially high F1. Therefore, `response` is excluded from the split files entirely.

**Trap 2 — `flags` leakage:** `flags` describe how the dataset was generated, not information available in a real ticket. Using them creates **train/serving skew**: the model performs well in testing but cannot rely on the same information in production.



In [3]:
# The one-line version of trap 1, so the size difference is visible:
print("mean length of `instruction`:", int(df_raw["instruction"].str.len().mean()), "chars")
print("mean length of `response`   :", int(df_raw["response"].str.len().mean()), "chars")
print("\nrows per intent - min:", df_raw["intent"].value_counts().min(),
      "max:", df_raw["intent"].value_counts().max())
print("\nthe input is defined explicitly, never by subtraction:")
print("    X = df['instruction']      y = df['intent']")

mean length of `instruction`: 46 chars
mean length of `response`   : 634 chars

rows per intent - min: 950 max: 1000

the input is defined explicitly, never by subtraction:
    X = df['instruction']      y = df['intent']


### The `flags` Column — Discrepancy Between Documentation and the Actual Data

The dataset card documents 12 generation tags: `M L B I C N P Q W K E Z`. However, a direct inspection of the dataset file revealed 14 distinct letters. Two of them, `S` and `V`, are not documented anywhere, neither in the list of tags in use nor in the list of tags not in use.

This represents another discrepancy between the published documentation and the actual dataset. Therefore, the analysis of the `flags` column was based on the values observed directly in the dataset file rather than solely on the tag inventory provided in the dataset card.


In [4]:
from collections import Counter

flag_counts = Counter("".join(df_raw["flags"].astype(str)))
documented = set("MLBICNPQWKEZ")
tbl = pd.DataFrame(
    [{"flag": f, "rows": n, "pct_of_dataset": round(100 * n / len(df_raw), 2),
      "documented": f in documented} for f, n in flag_counts.most_common()])
display(tbl)

for f in sorted(set(flag_counts) - documented):
    sub = df_raw[df_raw["flags"].str.contains(f, regex=False)]
    print(f"\nundocumented flag '{f}': {len(sub)} rows, intents = {sorted(sub['intent'].unique())}")
    print("   example:", sub["instruction"].iloc[0])

,flag,rows,pct_of_dataset,documented
0,B,26872,100.00,True
1,L,24115,89.74,True
2,Q,8968,33.37,True
3,I,7839,29.17,True
4,Z,5286,19.67,True
5,M,4920,18.31,True
6,C,2646,9.85,True
7,K,2227,8.29,True
8,E,1882,7.00,True
9,P,1329,4.95,True



undocumented flag 'S': 417 rows, intents = ['delivery_options']
   example: do you ship to {{Delivery City}}?

undocumented flag 'V': 77 rows, intents = ['cancel_order', 'check_invoice', 'get_refund', 'recover_password']
   example: I cannot pay for the last purchase I have made, cancel it


### The category list in the file vs the published documentation

The dataset card claims *"27 intents assigned to 10 categories"* and lists 21 intents under 10 headings. The file has 27 intents under 11 categories, and three assumed category names were also incorrect (`SHIPPING_ADDRESS`, `CANCELLATION_FEE`, `NEWSLETTER` are actually `SHIPPING`, `CANCEL`, `SUBSCRIPTION`).

This is exactly why `intent2cat.json` is **built from the DataFrame and never typed in from documentation**.

In [5]:
mapping_check = df_raw.groupby("intent")["category"].nunique()
print("intents mapping to more than one category:", int((mapping_check > 1).sum()), "(must be 0)")
print("\ncategories actually in the file:", sorted(df_raw['category'].unique()))
print("\nintents per category:")
print(df_raw.groupby("category")["intent"].nunique().sort_values(ascending=False).to_string())

intents mapping to more than one category: 0 (must be 0)

categories actually in the file: ['ACCOUNT', 'CANCEL', 'CONTACT', 'DELIVERY', 'FEEDBACK', 'INVOICE', 'ORDER', 'PAYMENT', 'REFUND', 'SHIPPING', 'SUBSCRIPTION']

intents per category:
category
ACCOUNT         6
ORDER           4
REFUND          3
INVOICE         2
CONTACT         2
FEEDBACK        2
DELIVERY        2
SHIPPING        2
PAYMENT         2
CANCEL          1
SUBSCRIPTION    1


## 1.2 - The finding: measure the leakage *before* cleaning anything

### What "leakage" means here

The dataset contains many **near-duplicate sibling examples** generated from the same base templates. With a naive random split, siblings can appear in both train and test, making evaluation overly optimistic: the model is tested on examples very similar to ones it has already seen.

The leakage is therefore **measured before cleaning**, by comparing the naive split with a family-aware clean split.

### How similarity is measured

Similarity is computed using **TF-IDF character n-grams (3–5)** and **cosine similarity**. Character n-grams are used because they remain effective with the dataset's deliberate typos (e.g. `delivery` vs. `deliverly`).

The **same fitted vector space** is reused for clustering and leakage verification, so all similarity thresholds are measured in the same space.

In [6]:
# Whitespace normalisation first: a double space is a typing artefact, and
# without collapsing it two identical sentences look different to every
# comparison that follows.
df_ws = D.drop_empty_rows(D.normalise_whitespace(df_raw))
print(f"rows after whitespace normalisation / empty-row removal: {len(df_ws)} (from {len(df_raw)})")

# A vector space over the data as it is now - BEFORE any de-duplication.
vec_before, X_before = D.build_vector_space(df_ws["instruction"])
print("vocabulary size:", len(vec_before.vocabulary_), "character n-grams")

pos_before = np.arange(len(df_ws))
df_ws = df_ws.assign(pos=pos_before)

rows after whitespace normalisation / empty-row removal: 26872 (from 26872)


vocabulary size: 10910 character n-grams


#### Does whitespace normalisation earn its keep?

In [7]:
# Does the one unconditional cleaning step earn its keep?
#
# The row count says no: 26,872 before and 26,872 after, which is exactly the
# kind of step that could be challenged as cleaning for its own sake. The
# answer is that it is not there to remove rows. It is there so that the next
# step can recognise duplicates it would otherwise walk straight past.
ws_impact = D.whitespace_impact(df_raw)
for k, v in ws_impact.items():
    print(f"{k:46}: {v:,}")

print()
print(f"Collapsing whitespace rewrites {ws_impact['texts_changed']} texts, and that lets "
      f"{ws_impact['extra_pairs_merged']} additional")
print("(instruction, intent) pairs be recognised as exact duplicates "
      f"({ws_impact['exact_duplicates_found_with_normalisation']:,} against "
      f"{ws_impact['exact_duplicates_found_without_normalisation']:,}).")
print("Those pairs differ only by a double space. Without this step they are two")
print("distinct rows that can land on opposite sides of the split and leak.")
# (these scalars are not written on their own; they are folded into
#  corpus_cleaning_funnel.csv further down, beside the stage they explain)

rows_in                                       : 26,872
texts_changed                                 : 551
empty_rows_dropped                            : 0
exact_duplicates_found_without_normalisation  : 2,237
exact_duplicates_found_with_normalisation     : 2,318
extra_pairs_merged                            : 81

Collapsing whitespace rewrites 551 texts, and that lets 81 additional
(instruction, intent) pairs be recognised as exact duplicates (2,318 against 2,237).
Those pairs differ only by a double space. Without this step they are two
distinct rows that can land on opposite sides of the split and leak.


In [8]:
# The naive split: stratified 70/15/15, no clustering, no family awareness.
tr_b, va_b, te_b = D.split_stratified(df_ws, seed=D.SPLIT_SEED)
print(f"naive split on the un-deduplicated data: {len(tr_b)} / {len(va_b)} / {len(te_b)}")

sim_before = D.max_similarity_to_train(X_before, tr_b["pos"].values, te_b["pos"].values)
leak_before = D.leakage_summary(sim_before, D.exact_overlap(tr_b, te_b))

print("\n--- LEAKAGE IN A NAIVE SPLIT, BEFORE ANY CLEANING ---")
for k, v in leak_before.items():
    print(f"  {k:>24} : {v}")

naive split on the un-deduplicated data: 18810 / 4031 / 4031



--- LEAKAGE IN A NAIVE SPLIT, BEFORE ANY CLEANING ---
                 exact_pct : 10.49
               ge_0.95_pct : 30.64
               ge_0.90_pct : 52.02
               ge_0.80_pct : 82.71
     median_max_similarity : 0.906


> In a plain random split, **10.49%** of test rows appear verbatim in training, and **52.02%** are within 0.10 cosine similarity of a training row. Therefore, the random split does not provide a clean measure of generalisation.

### Label conflicts

No exact sentence appears with two different intents (**0 exact conflicts**), so there is no theoretical accuracy ceiling caused by contradictory duplicate labels.

However, **near-identical sentences with different intents do exist**. These are retained because they are not duplicates; they represent genuine ambiguity between intents and explain why perfect classification may not be achievable.


In [9]:
conflicts = D.find_label_conflicts(df_ws)
print("rows involved in an exact label conflict:", len(conflicts))
print("distinct conflicting sentences        :", conflicts['instruction'].nunique())

rows involved in an exact label conflict: 0
distinct conflicting sentences        : 0


In [10]:
# The soft version of the same question - and unlike the exact version, it is
# not empty.
#
# An earlier version of this measurement used a 3,000-row sample. The estimate
# was accurate (1.23% sampled against 1.21% true), so sampling was not the
# issue - but the exhaustive scan takes about twenty seconds, so there is no
# reason to estimate something that can be computed directly.
twins_ws = D.cross_intent_neighbours(df_ws, X_before)

print(f"rows whose nearest neighbour is >= {D.NEAR_DUP_THRESHOLD} similar but carries a "
      f"DIFFERENT intent:")
print(f"  {len(twins_ws):,} of {len(df_ws):,} rows = {100*len(twins_ws)/len(df_ws):.2f}%")
print("  (this frame still contains exact duplicates; the same measurement on the")
print("   deduplicated corpus the split is drawn from appears further down)")
print()

for r in twins_ws.head(5).itertuples():
    print(f"  sim={r.similarity}")
    print(f"    [{r.intent}] {r.instruction}")
    print(f"    [{r.twin_intent}] {r.twin_instruction}")

print()
print("These are NOT errors to clean away. No classifier can be right about both")
print("members of such a pair, so they are a CEILING on achievable accuracy. Deleting")
print("them would be removing the hard cases to make the score look better, which is")
print("the exact failure this project exists to avoid.")
# The five pairs above are what is worth keeping. The full 326-row frame was a
# dump of rows rather than a finding, so it is measured here and not written out.

rows whose nearest neighbour is >= 0.9 similar but carries a DIFFERENT intent:
  326 of 26,872 rows = 1.21%
  (this frame still contains exact duplicates; the same measurement on the
   deduplicated corpus the split is drawn from appears further down)

  sim=0.96
    [registration_problems] help creating user
    [create_account] creating user
  sim=0.96
    [create_account] creating user
    [registration_problems] help creating user
  sim=0.959
    [create_account] creating account
    [registration_problems] help creating account
  sim=0.959
    [registration_problems] help creating account
    [create_account] creating account
  sim=0.947
    [check_invoice] I want assistance to see my invoices from {{Person Name}}
    [get_invoice] I want assistance to get my invoices from {{Person Name}}

These are NOT errors to clean away. No classifier can be right about both
members of such a pair, so they are a CEILING on achievable accuracy. Deleting
them would be removing the hard cases to 

## 1.3 - Cleaning

Minimal text cleaning was applied: repeated whitespace was collapsed, empty rows were removed, and UTF-8 encoding was preserved.

The original text was otherwise preserved to retain meaningful linguistic cues and ensure consistency between training and serving.


In [11]:
# drop_duplicates() with no `subset` compares every column, including `flags`
# and `response`, so it silently keeps rows that are exact text duplicates.
# This cell demonstrates that failure before using the correct subset.
no_subset = len(df_ws.drop_duplicates())                 # compares EVERY column
with_subset = len(D.drop_exact_duplicates(df_ws))        # compares instruction + intent

print(f"rows before de-duplication                : {len(df_ws)}")
print(f"drop_duplicates(), default (all columns)  : removes {len(df_ws) - no_subset} rows")
print(f"subset=[instruction, intent], correct     : removes {len(df_ws) - with_subset} rows")
print()
print("The default also compares `flags` and `response`. Two rows with the same")
print("instruction text but a different flag string are treated as distinct, so")
print("nothing is removed. Restricting the subset to `instruction` and `intent`")
print("is what correctly defines an exact duplicate here.")

rows before de-duplication                : 26872
drop_duplicates(), default (all columns)  : removes 0 rows
subset=[instruction, intent], correct     : removes 2318 rows

The default also compares `flags` and `response`. Two rows with the same
instruction text but a different flag string are treated as distinct, so
nothing is removed. Restricting the subset to `instruction` and `intent`
is what correctly defines an exact duplicate here.


In [12]:
df_dedup = D.drop_exact_duplicates(df_ws).reset_index(drop=True)
df_dedup["pos"] = np.arange(len(df_dedup))
print(f"{len(df_raw)} raw -> {len(df_dedup)} after exact de-duplication "
      f"({100*len(df_dedup)/len(df_raw):.1f}% kept)")

26872 raw -> 24554 after exact de-duplication (91.4% kept)


### Near-duplicates: cluster rather than delete

Near-duplicate rows were grouped into **families** using character n-gram similarity and connected components. Each family was kept within a single split to prevent leakage between train, validation, and test.

Similarity was computed separately within each intent for efficiency, and the existing fitted vector space was refitted on the deduplicated dataset before clustering.


In [13]:
vec, X = D.build_vector_space(df_dedup["instruction"])
print("vector space:", X.shape[0], "rows x", X.shape[1], "character n-grams,",
      f"{X.nnz:,} non-zeros ({100*X.nnz/(X.shape[0]*X.shape[1]):.2f}% dense)")

vector space: 24554 rows x 10579 character n-grams, 2,183,015 non-zeros (0.84% dense)


### Choosing the near-duplicate threshold

Clustering needs exactly one number: the cosine similarity above which two sentences are declared siblings of the same template. Everything above it is grouped into one family, and a family always stays on one side of the split.

**The number can fail in two opposite directions, and neither failure announces itself:**

| set it too **high** | set it too **low** |
| --- | --- |
| Only near-identical twins get grouped. Real siblings stay loose, land on opposite sides of the split, and the test score then measures memorisation instead of generalisation. | Sentences that merely share a topic get welded into one family. Each family contributes a single row, so training data is deleted in exchange for nothing. |

**So the rule is fixed first, before any candidate is measured:**

> **Take the loosest threshold - the one that keeps the most training data - at which the split is still clean.**
> *Clean* means: fewer than 1% of held-out rows are left sitting within 0.10 cosine of a training row.

*Loosest*, because the training set is the product and every step stricter deletes rows from it. *Still clean*, because a split that leaks is not worth any quantity of data. The two halves pull in opposite directions, which is what makes this a rule and not a preference.

The two tables below apply that rule to six candidates, bracketing the answer from both sides:

1. the first prices each candidate in **data kept** - how many families survive, and therefore how many training rows;
2. the second prices the same candidates in **leakage left behind** - build the split each threshold would actually produce, then measure how much test<->train similarity survives it.

Both are needed. The family count alone never says whether the cleaning worked, and the leakage column on its own would happily recommend deleting the entire corpus.

**How to read the leakage column**, because it is the kind of thing an examiner should catch: it is partly tautological. Clustering at threshold *t* removes same-intent pairs above *t* by construction, so of course a split cleaned at *t* has almost nothing left above *t*. What the column really shows is the **failure** case - where the cliff sits - and here the cliff is sharp: one step above the selected value the residual jumps roughly twentyfold. The informative comparison is between that column and `train_rows_per_intent`, and it is what settles the strict end: the two strictest candidates come out no cleaner than the selected one and cost 40% and 68% of the training data respectively.

In [14]:
scan = D.threshold_scan(X, df_dedup["intent"].values)
display(scan)


,threshold,n_groups,retained_pct,mean_family_size
0,0.80,4582,18.7,5.36
1,0.85,8608,35.1,2.85
2,0.90,14133,57.6,1.74
3,0.91,15423,62.8,1.59
4,0.92,16635,67.7,1.48
5,0.95,19879,81.0,1.24


In [15]:
# For each candidate: build the clean split it would produce and measure how
# much test<->train similarity survives it. This is the second half of what the
# rule needs - the family count alone does not say whether the cleaning worked.
rows = []
for t in [0.80, 0.85, 0.90, 0.91, 0.92, 0.95]:
    g = D.near_duplicate_groups(X, df_dedup["intent"].values, threshold=t)
    reps = D.pick_representatives(df_dedup.assign(dup_group=g))
    tr, va, te = D.split_stratified(reps, seed=D.SPLIT_SEED)
    s = D.max_similarity_to_train(X, tr["pos"].values, te["pos"].values)
    rows.append({"threshold": t, "rows_kept": len(reps), "train_rows": len(tr),
                 "train_rows_per_intent": round(len(tr) / 27),
                 "test>=0.95_pct": round(100 * float((s >= 0.95).mean()), 2),
                 "test>=0.90_pct": round(100 * float((s >= 0.90).mean()), 2),
                 "median_max_sim": round(float(np.median(s)), 3)})
threshold_tradeoff = pd.DataFrame(rows)
display(threshold_tradeoff)

# The rule from the markdown above, executed against the table rather than
# restated. "Clean" is its operational definition: under 1% of held-out rows
# left within 0.10 cosine of a training row. Among the candidates that clear
# that bar, the loosest is the answer, because it is the one that keeps most.
CLEAN_BAR_PCT = 1.0
clean_enough = threshold_tradeoff[threshold_tradeoff["test>=0.90_pct"] < CLEAN_BAR_PCT]
selected = float(clean_enough["threshold"].max())
picked = threshold_tradeoff.loc[threshold_tradeoff["threshold"] == selected].iloc[0]

print(f"candidates that stay clean (< {CLEAN_BAR_PCT}% of test rows within 0.10 cosine): "
      f"{list(clean_enough['threshold'])}")
print(f"the loosest of those, and so the rule's answer: {selected}")
print(f"what it keeps: {int(picked['train_rows']):,} training rows, "
      f"{int(picked['train_rows_per_intent'])} per intent")
print(f"what it leaves: {picked['test>=0.90_pct']}% of test rows within 0.10 cosine of train")

# The constant in src/data.py is NOT an opinion typed into a config file: it is
# what the rule above returns from the two tables on screen. Change the rule,
# the bar or the candidate list and this line is what complains.
assert selected == D.NEAR_DUP_THRESHOLD, (
    f"the rule selected {selected} but src.data.NEAR_DUP_THRESHOLD is "
    f"{D.NEAR_DUP_THRESHOLD} - the frozen split and the derivation disagree")

# One table per key, not one table per cell: the family counts from the scan
# above and the leakage measurements here describe the same six candidates.
assert (scan["n_groups"].values == threshold_tradeoff["rows_kept"].values).all()
threshold_choice = scan.merge(threshold_tradeoff.drop(columns=["rows_kept"]),
                              on="threshold", validate="1:1")
threshold_choice.to_csv(METRICS_DIR / "corpus_dedup_threshold.csv", index=False)

,threshold,rows_kept,train_rows,train_rows_per_intent,test>=0.95_pct,test>=0.90_pct,median_max_sim
0,0.80,4582,3207,119,0.00,0.15,0.655
1,0.85,8608,6025,223,0.00,0.46,0.740
2,0.90,14133,9893,366,0.05,0.24,0.809
3,0.91,15423,10796,400,0.00,5.06,0.824
4,0.92,16635,11644,431,0.04,10.46,0.838
5,0.95,19879,13915,515,0.00,30.32,0.865


candidates that stay clean (< 1.0% of test rows within 0.10 cosine): [0.8, 0.85, 0.9]
the loosest of those, and so the rule's answer: 0.9
what it keeps: 9,893 training rows, 366 per intent
what it leaves: 0.24% of test rows within 0.10 cosine of train


In [17]:
groups = D.near_duplicate_groups(X, df_dedup["intent"].values, threshold=D.NEAR_DUP_THRESHOLD)
df_dedup["dup_group"] = groups
n_groups = df_dedup["dup_group"].nunique()
sizes = df_dedup["dup_group"].value_counts()

print(f"families at threshold {D.NEAR_DUP_THRESHOLD}: {n_groups:,} "
      f"({100*n_groups/len(df_dedup):.1f}% of {len(df_dedup):,} rows)")
print(f"family sizes - mean {sizes.mean():.2f}, median {int(sizes.median())}, max {sizes.max()}")
print("\nfamily size distribution:")
print(sizes.value_counts().sort_index().head(8).to_string())

families at threshold 0.9: 14,133 (57.6% of 24,554 rows)
family sizes - mean 1.74, median 1, max 117

family size distribution:
count
1    11100
2     1498
3      546
4      285
5      167
6      137
7       76
8       71


In [18]:
# What a family actually looks like - the argument condensed into one example.
biggest = sizes.index[0]
fam = df_dedup[df_dedup["dup_group"] == biggest]
print(f"family #{biggest} - intent '{fam['intent'].iloc[0]}', {len(fam)} rows:\n")
for t, f in zip(fam["instruction"].head(10), fam["flags"].head(10)):
    print(f"  [{f:<8}] {t}")

family #8619 - intent 'newsletter_subscription', 117 rows:

  [BLZ     ] I try tro subscribe to the newsletter
  [BL      ] I can't unsubscribe from your company newsletter
  [BL      ] I am trying to unsubscribe from your company newsletter
  [BL      ] I try to unsubscribe to your company newsletter
  [BL      ] assistance to unsubscribe from the company newsletter
  [BLP     ] I don't know how I could unsubscribe to the newsletter
  [BL      ] I want help to unsubscribe to the company newsletter
  [BLQ     ] i need help to unsubscribe from ur company newsletter
  [BL      ] I have got to unsubscribe to the newsletter
  [BIL     ] can you help me unsubscribe to the newsletter?


### Stress-testing the selected threshold: "is it too low?"

A rule that returns a number still owes the reader a look at what that number does to the actual sentences. The family above is the reason this check exists. Read top to bottom it mixes two
opposite requests - *subscribe* and *unsubscribe* - and the natural reaction is that
the threshold must be too low: surely 0.90 declared "subscribe" and "unsubscribe"
near-duplicates of each other?

Measured, that is not what happened. The direct similarity between the subscribe rows
and the unsubscribe rows sits far below the threshold; the two camps are welded
together by a handful of **bridge pairs** - sentence pairs that differ by exactly the
two characters `un` - and by the transitive closure of `connected_components`. The
cell below counts them.


In [19]:
# Inside the big family: direct similarity between the two "camps".
# `fam` comes from the cell above; `pos` indexes the shared vector space X.
fam_idx = fam["pos"].values
fam_txt = fam["instruction"].values
S_fam = (X[fam_idx] @ X[fam_idx].T).toarray()
np.fill_diagonal(S_fam, 0.0)

is_unsub = np.array(["unsub" in t.lower() for t in fam_txt])
cross = S_fam[np.ix_(~is_unsub, is_unsub)]        # subscribe rows x unsubscribe rows
bridges = int((cross >= D.NEAR_DUP_THRESHOLD).sum())

print(f"rows phrased as subscribe: {(~is_unsub).sum()}, as unsubscribe: {is_unsub.sum()}")
print(f"cross-camp pairs >= {D.NEAR_DUP_THRESHOLD}: {bridges} of {cross.size:,} "
      f"- these bridges weld the two camps into one family")
i, j = np.unravel_index(cross.argmax(), cross.shape)
print(f"\nstrongest bridge (sim={cross.max():.3f}):")
print("   ", fam_txt[~is_unsub][i])
print("   ", fam_txt[is_unsub][j])


rows phrased as subscribe: 32, as unsubscribe: 85
cross-camp pairs >= 0.9: 4 of 2,720 - these bridges weld the two camps into one family

strongest bridge (sim=0.910):
    need assistance to subscribe to the company newsletter
    i need assistance to unsubscribe to the company newsletter


Character n-grams cannot know that `un-` inverts a meaning: "...to subscribe to the
company newsletter" and "...to unsubscribe to the company newsletter" share nearly
every 3-5-gram. A few such pairs out of ~2,700 are enough, because a family is a
*connected component* - one edge merges two whole clusters. That is not a bug in the
clustering; it is the price of the transitive closure that prevents A~B~C leakage
(design point 3 above), and the price is worth paying.

**Raising the threshold to make the merge go away costs the project its finding.**
The trade-off table above already showed 0.95 failing (30% of test within 0.10 cosine
of train); the cell below adds the extreme case:


In [20]:
# The extreme end of the trade-off: threshold 0.99.
g99 = D.near_duplicate_groups(X, df_dedup["intent"].values, threshold=0.99)
reps99 = D.pick_representatives(df_dedup.assign(dup_group=g99))
tr99, _, te99 = D.split_stratified(reps99, seed=D.SPLIT_SEED)
s99 = D.max_similarity_to_train(X, tr99["pos"].values, te99["pos"].values)

print(f"at 0.99, {len(reps99):,} of {len(df_dedup):,} rows survive as their own family "
      f"({100*len(reps99)/len(df_dedup):.1f}%) - clustering is effectively switched off")
print(f"test rows within 0.10 cosine of a train row: {100*(s99 >= 0.90).mean():.2f}%")
print(f"for reference, the naive split before any cleaning: {leak_before['ge_0.90_pct']}%")
print("\nraising the threshold to chase the subscribe/unsubscribe merge walks the split")
print(f"most of the way back to naive - which is why no candidate this loose "
      f"clears the\nclean bar, and why the rule stopped at {selected}.")


at 0.99, 23,697 of 24,554 rows survive as their own family (96.5%) - clustering is effectively switched off
test rows within 0.10 cosine of a train row: 45.79%
for reference, the naive split before any cleaning: 52.02%

raising the threshold to chase the subscribe/unsubscribe merge walks the split
most of the way back to naive - which is why no candidate this loose clears the
clean bar, and why the rule stopped at 0.9.


### What the family actually exposed: the taxonomy is coarser than the user's goal

The subscribe/unsubscribe pair is not a clustering artefact to fix - it is a property
of the dataset's label scheme, and it is measurable. Per intent: cluster the
**responses** into two groups (the NLG engine wrote different reply templates for
different goals), then ask how well the **instruction alone** predicts the response
cluster. If the customer's own words reliably signal a distinction that the intent
label ignores, that label is bundling more than one goal. `lift` is accuracy minus
the majority baseline; k=2 makes it a lower bound, not an exact measurement.

`newsletter_subscription` turns out to be near the top but **not an outlier**:
`complaint` (file a claim vs make a complaint), `switch_account` (upgrade tier vs
switch user) and `recover_password` (reset PIN vs recover access key) carry the same
structure.

This is report material, and it is a **limitation rather than a defect**. Nothing here
costs accuracy: the classifier's contract is to return the dataset's label, and for
both camps that label is identical, so a model can be perfectly correct on every one of
these rows. What the measurement says is that a correct intent does not identify what
the customer actually wanted - the customer's own wording carries a distinction the
label throws away. Any use of this classifier that treats one intent as one answer
inherits that gap, and the limitations section is where it gets stated.


In [21]:
# Per intent: can the instruction alone predict which of two response clusters
# a row belongs to? lift >> 0 means the label bundles distinguishable goals.
separability = D.subgoal_separability(df_dedup)
display(separability)
print(f"intents with lift >= 0.15: {(separability['lift'] >= 0.15).sum()} of 27")
rank = int(separability.index[separability["intent"] == "newsletter_subscription"][0]) + 1
print(f"newsletter_subscription ranks #{rank} - the family that raised the alarm is "
      f"not even the most bundled intent")


,intent,n_rows,smaller_cluster_pct,instruction_predicts_cluster,majority_baseline,lift
0,complaint,998,39.6,0.978,0.604,0.374
1,newsletter_subscription,999,43.0,0.940,0.570,0.370
2,recover_password,994,35.5,0.998,0.645,0.353
3,get_invoice,906,49.7,0.855,0.503,0.352
4,delivery_options,658,41.3,0.895,0.587,0.309
5,check_payment_methods,998,40.9,0.836,0.591,0.244
6,registration_problems,998,30.4,0.940,0.696,0.244
7,check_invoice,920,37.4,0.849,0.626,0.223
8,create_account,889,33.6,0.882,0.664,0.218
9,payment_issue,998,28.2,0.921,0.718,0.202


intents with lift >= 0.15: 18 of 27
newsletter_subscription ranks #2 - the family that raised the alarm is not even the most bundled intent


### One more design temptation, measured and declined: train on ALL family members

If families are the unit, why collapse them at all? The alternative ("group split")
assigns whole families to one side and keeps every member: train grows from ~9.9k to
~17.3k rows at the same leakage. Tempting - but the extra rows are not new
information, they are re-weighting:


In [22]:
# `sizes` was computed above: rows per family at the working threshold.
top1pct_rows = sizes.sort_values(ascending=False).head(int(0.01 * len(sizes))).sum()
print(f"families: {len(sizes):,}   of which singletons: {(sizes == 1).sum():,} "
      f"({100 * (sizes == 1).mean():.1f}%)")
print(f"the top 1% largest families hold {100 * top1pct_rows / len(df_dedup):.1f}% of all rows")
print(f"largest family: {sizes.max()} rows -> its template would weigh {sizes.max()}x a singleton")


families: 14,133   of which singletons: 11,100 (78.5%)
the top 1% largest families hold 15.5% of all rows
largest family: 117 rows -> its template would weigh 117x a singleton


Training on all members weights each *template* by how many variants the generator
happened to emit - up to 117x - while the test set (one representative per family)
stays template-uniform. That is a train/test distribution mismatch bought with the
extra rows. The collapsed design asks one coherent question - *does the model
generalise across templates?* - with train and test in the same units; ~366 rows per
intent is ample for a 27-class problem; and every GPU run is ~40% cheaper.

**Verdict of the whole review: threshold 0.90 stays, collapse-then-split stays.**
What *does* change: the ~10k non-representative rows stop being discarded. Dropping 42%
of the corpus with no record of what went, which family it belonged to, or which side
that family landed on is not a decision anyone can audit afterwards - so they get an
explicit home: `full_corpus.csv`, built in section 1.5b below. That file is also what
makes the collapse's cost measurable instead of assumed: `src.data.train_side_rows()`
rebuilds the uncollapsed training set out of it, and `02_baselines.ipynb` scores it.
(Training with `sample_weight = 1/family_size`, or capped at k rows per family, would
sit between the two designs; both remain honest ablations that were not run.)


### Which row represents the family?

One row per family survives. The representative is drawn at random with the frozen split seed.

In [23]:
reps_random = D.pick_representatives(df_dedup, seed=D.SPLIT_SEED)

### The before/after table

Four rows, tracking the corpus through each cleaning stage. Plus per-intent retention, because an aggregate number can hide one intent collapsing to nothing.

In [24]:
before_after = pd.DataFrame([
    {"stage": "raw file",                       "rows": len(df_raw)},
    {"stage": "after whitespace + empty rows",  "rows": len(df_ws)},
    {"stage": "after exact duplicates",         "rows": len(df_dedup)},
    {"stage": f"after family clustering @ {D.NEAR_DUP_THRESHOLD}", "rows": len(reps_random)},
])
before_after["pct_of_raw"] = (100 * before_after["rows"] / len(df_raw)).round(1)
display(before_after)

retention = (100 * reps_random["intent"].value_counts() /
             df_raw["intent"].value_counts()).round(1).sort_values()
print("per-intent retention %, lowest and highest five:")
print(pd.concat([retention.head(5), retention.tail(5)]).to_string())
# Two files written here, one per key rather than one per cell.
#   corpus_cleaning_funnel.csv - rows surviving each stage, with the whitespace
#     scalars from further up attached to the stage they actually explain.
#   corpus_per_intent.csv      - retention joined to the subgoal separability
#     measured earlier; both are keyed by intent, so both belong in one table.
funnel = before_after.copy()
i_ws = funnel.index[funnel["stage"].str.startswith("after whitespace")][0]
i_dup = funnel.index[funnel["stage"].str.startswith("after exact")][0]
for col, where, value in [
    ("texts_changed",      i_ws,  ws_impact["texts_changed"]),
    ("empty_rows_dropped", i_ws,  ws_impact["empty_rows_dropped"]),
    ("extra_pairs_merged", i_ws,  ws_impact["extra_pairs_merged"]),
    ("exact_dups_found",   i_dup, ws_impact["exact_duplicates_found_with_normalisation"]),
]:
    funnel[col] = pd.Series({where: value}, dtype="Int64").reindex(funnel.index)
funnel.to_csv(METRICS_DIR / "corpus_cleaning_funnel.csv", index=False)

per_intent = separability.merge(
    retention.rename("retention_pct").rename_axis("intent").reset_index(),
    on="intent", validate="1:1")
per_intent.to_csv(METRICS_DIR / "corpus_per_intent.csv",
                  index=False,
                  columns=["intent", "n_rows", "retention_pct", "smaller_cluster_pct",
                           "instruction_predicts_cluster", "majority_baseline", "lift"])


,stage,rows,pct_of_raw
0,raw file,26872,100.0
1,after whitespace + empty rows,26872,100.0
2,after exact duplicates,24554,91.4
3,after family clustering @ 0.9,14133,52.6


per-intent retention %, lowest and highest five:
intent
cancel_order               20.1
track_order                26.0
newsletter_subscription    33.3
track_refund               35.7
get_invoice                36.7
edit_account               67.0
change_shipping_address    69.7
review                     74.4
recover_password           79.7
place_order                81.9


## 1.4 - Split, twice, and freeze

| set | share | when it is touched |
|---|---|---|
| train | 70% | constantly - the model learns from it |
| val | 15% | every check during tuning, as often as needed |
| test | 15% | **once**, at the very end |

`train_test_split` has no three-way mode, so it runs twice: 70/30, then the 30 cut in half.

**Stratify on `intent` (27), not on `category` (11).** Stratifying on categories does not guarantee that every one of the 27 intents appears in val; a missing intent makes macro-F1 average 26 classes instead of 27, and nothing complains.

**Two seeds that are both called "seed".** The split seed (42) is frozen here forever. The *training* seed is a different constant, in a different file, and it holds the same value - this project runs everything at 42 and never at a second value. They keep separate names because they decide different things: confusing them produces variance that looks like training noise and is really different data, and sharing a value does not make them the same knob.

### Two full splits

- `clean/` respects the families - **this is the real split**, everything is built on it.
- `naive/` is the ordinary random split with no clustering - **the control group**, which is what later proves how much the leakage inflated the score.

`naive/` is built from the same exactly-de-duplicated frame. Exact-duplicate removal is standard practice; the variable under test is family awareness, and only that.

In [25]:
clean_tr, clean_va, clean_te = D.split_stratified(reps_random, seed=D.SPLIT_SEED)
naive_tr, naive_va, naive_te = D.split_stratified(df_dedup,   seed=D.SPLIT_SEED)

print(f"clean : {len(clean_tr)} / {len(clean_va)} / {len(clean_te)}   (total {len(reps_random)})")
print(f"naive : {len(naive_tr)} / {len(naive_va)} / {len(naive_te)}   (total {len(df_dedup)})")

# The check that stratification actually did its job: all 27 intents present
# in every one of the six files.
for name, part in [("clean/train", clean_tr), ("clean/val", clean_va), ("clean/test", clean_te),
                   ("naive/train", naive_tr), ("naive/val", naive_va), ("naive/test", naive_te)]:
    assert part["intent"].nunique() == 27, f"{name} is missing an intent!"
print("\nall six splits contain all 27 intents.")
print(f"smallest intent in clean/val: {clean_va['intent'].value_counts().min()} rows")

clean : 9893 / 2120 / 2120   (total 14133)
naive : 17187 / 3683 / 3684   (total 24554)

all six splits contain all 27 intents.
smallest intent in clean/val: 30 rows


### Verify that the clean split is actually clean

The claim is "no test row sits within 0.10 cosine of a training row". It is only worth making if it is measured, in the same vector space that was used to do the cleaning.

The residual that survives is the interesting part: it is **cross-intent** similarity - near-identical sentences carrying different labels. That is not leakage, it is the semantic-conflict phenomenon from above, and removing it would destroy exactly the hard cases the model should be judged on.

The **same-intent** half is not measured separately here. Everything at or above the threshold was collapsed into one family, and a family never sits on two sides of the split, so that number is fixed at zero before any measurement is taken - a decision, not a finding. It is reported further down anyway, in a feature space that had no part in building the split, where it was free to come out non-zero.

In [26]:
sim_clean = D.max_similarity_to_train(X, clean_tr["pos"].values, clean_te["pos"].values)
sim_naive = D.max_similarity_to_train(X, naive_tr["pos"].values, naive_te["pos"].values)

leak_clean = D.leakage_summary(sim_clean, D.exact_overlap(clean_tr, clean_te))
leak_naive = D.leakage_summary(sim_naive, D.exact_overlap(naive_tr, naive_te))

leak_table = pd.DataFrame({
    "naive (before cleaning, incl. exact dups)": leak_before,
    "naive (committed control split)": leak_naive,
    "clean (committed real split)": leak_clean,
}).T
display(leak_table)

,exact_pct,ge_0.95_pct,ge_0.90_pct,ge_0.80_pct,median_max_similarity
"naive (before cleaning, incl. exact dups)",10.49,30.64,52.02,82.71,0.906
naive (committed control split),0.00,23.97,47.29,80.75,0.894
clean (committed real split),0.00,0.05,0.24,55.05,0.809


#### The ambiguity ceiling, measured exhaustively

In [28]:
# The cross-intent residual in the table above is not an isolated curiosity.
# Measured exhaustively over the deduplicated corpus that the split is actually
# drawn from:
twins = D.cross_intent_neighbours(df_dedup, X)
twin_pct = 100 * len(twins) / len(df_dedup)

print(f"rows with a cross-intent twin >= {D.NEAR_DUP_THRESHOLD}: "
      f"{len(twins):,} of {len(df_dedup):,} = {twin_pct:.2f}%")
print()

pair_counts = (twins.assign(pair=[" <-> ".join(sorted([a, b]))
                                  for a, b in zip(twins["intent"], twins["twin_intent"])])
                    ["pair"].value_counts())
print(f"the conflict is concentrated, not diffuse - {len(pair_counts)} intent pairs in total:")
print(pair_counts.to_string())
print()
print(f"{pair_counts.index[0]} alone accounts for "
      f"{100*pair_counts.iloc[0]/len(twins):.0f}% of every conflicted row.")
print()
print("The tempting next sentence is 'therefore accuracy cannot exceed "
      f"{1 - twin_pct/100:.4f}'.")
print("Do not write it. It is an assumption dressed as a measurement, and")
print("notebooks/02_baselines.ipynb tests it directly: only 6 of the 2,120 clean/val")
print("rows have a cross-intent twin at all - the family collapse removes most of them,")
print("because they cluster inside the big invoice families - and both TF-IDF baselines")
print("classify all 6 correctly. A row with a near-identical neighbour under a different")
print("label is not unclassifiable; the one token that differs is exactly the token a")
print("bag-of-words model keys on.")
print()
print("What this DOES measure is a property of the taxonomy: the label scheme separates")
print(f"intents on distinctions fine enough that {twin_pct:.2f}% of the corpus sits one word")
print("away from a different label. That is a caution about paraphrase robustness, and the")
print("companion to the subgoal-separability finding above - this taxonomy is fine where")
print("the wording is and coarse where the goal is. It is not a bound on the score.")
print("An earlier summary line stating 'no measurable labelling ceiling from ambiguity'")
print("was incorrect: there IS measurable ambiguity, it just does not cost")
print("accuracy on this split.")
# What is saved is a dozen examples, one per conflicting intent pair, because
# this is a claim about the taxonomy and a reader has to be able to check it.
# Not the 203-row frame: 02_baselines.ipynb recomputes what it needs through
# src.data rather than depending on an index file written here.
examples = (twins.assign(pair=[" <-> ".join(sorted([a, b])) for a, b
                               in zip(twins["intent"], twins["twin_intent"])])
                 .sort_values("similarity", ascending=False)
                 .groupby("pair", as_index=False).first()
                 .sort_values("similarity", ascending=False)
                 .head(12))
examples.to_csv(METRICS_DIR / "corpus_cross_intent_examples.csv", index=False,
                columns=["pair", "similarity", "instruction", "twin_instruction"])
print(f"saved {len(examples)} examples, one per pair, out of {len(twins)} conflicted rows")

print()
print("Five of them, which is the part worth keeping in the repository:")
for r in twins.head(5).itertuples():
    print(f"  sim={r.similarity}")
    print(f"    [{r.intent}] {r.instruction}")
    print(f"    [{r.twin_intent}] {r.twin_instruction}")

rows with a cross-intent twin >= 0.9: 203 of 24,554 = 0.83%

the conflict is concentrated, not diffuse - 7 intent pairs in total:
pair
check_invoice <-> get_invoice               148
get_refund <-> track_refund                  21
edit_account <-> switch_account              17
cancel_order <-> change_order                 6
create_account <-> switch_account             5
create_account <-> registration_problems      4
delete_account <-> switch_account             2

check_invoice <-> get_invoice alone accounts for 73% of every conflicted row.

The tempting next sentence is 'therefore accuracy cannot exceed 0.9917'.
Do not write it. It is an assumption dressed as a measurement, and
notebooks/02_baselines.ipynb tests it directly: only 6 of the 2,120 clean/val
rows have a cross-intent twin at all - the family collapse removes most of them,
because they cluster inside the big invoice families - and both TF-IDF baselines
classify all 6 correctly. A row with a near-identical neighbour under

#### The zero in the clustering space is a constraint, not a measurement

In [29]:
# The same-intent row of the table below reads 0.00%, and it is worth exactly as
# much as the space it is measured in - in THIS space it could not have come out
# any other way, which is what the `guaranteed_zero` column marks.
#
# Families are the connected components of the >= 0.90 similarity graph WITHIN
# one intent, and a whole family always lands on one side of the split. So a
# same-intent pair straddling the split at >= 0.90 would have to be one family
# on two sides, which cannot happen. The observed maximum is the giveaway: the
# distribution is truncated exactly at the threshold, which is what a
# constraint looks like, not what a measurement looks like.
#
# The remedy is not to weaken the claim. It is to re-measure it in a feature
# space that had no part in building the split, where the answer was free to be
# non-zero.
two_spaces = D.residual_leakage_two_spaces(clean_tr, clean_te, df_dedup)
display(two_spaces)
two_spaces.to_csv(METRICS_DIR / "corpus_residual_leakage.csv", index=False)

is_char = two_spaces["space"].str.startswith("char_wb")
is_same = two_spaces["relation"] == "same_intent"
char_max = two_spaces.loc[is_char & is_same, "max"].iloc[0]
word_residual = float(two_spaces.loc[~is_char & is_same, "ge_0.90_pct"].iloc[0])

print(f"largest same-intent similarity in the clustering space: {char_max}")
print(f"the clustering threshold                              : {D.NEAR_DUP_THRESHOLD}")
print(f"gap                                                   : "
      f"{D.NEAR_DUP_THRESHOLD - char_max:.6f}")
print()
print("The whole distribution stops immediately below the threshold, with nothing at")
print("all above it. Real measurements do not land that neatly against a round number")
print("chosen by hand; constraints do. This one is a constraint.")
print()
print(f"In the independent word(1,2) space the same residual is {word_residual}% - small,")
print("but it was free to have been large, which is what makes it evidence.")

,space,relation,guaranteed_zero,ge_0.95_pct,ge_0.90_pct,ge_0.80_pct,median,max
0,"char_wb(3,5)",same_intent,True,0.00,0.00,54.10,0.807,0.899984
1,"char_wb(3,5)",any_intent,False,0.05,0.24,55.05,0.809,0.961152
2,"word(1,2)",same_intent,False,0.38,1.79,16.23,0.692,1.000000
3,"word(1,2)",any_intent,False,0.38,1.79,16.51,0.695,1.000000


largest same-intent similarity in the clustering space: 0.899984
the clustering threshold                              : 0.9
gap                                                   : 0.000016

The whole distribution stops immediately below the threshold, with nothing at
all above it. Real measurements do not land that neatly against a round number
chosen by hand; constraints do. This one is a constraint.

In the independent word(1,2) space the same residual is 1.79% - small,
but it was free to have been large, which is what makes it evidence.


#### What are those word-space pairs, actually?

In [30]:
# The word-space residual is a percentage until you read the sentences behind
# it. Reading them is what turns it into a claim - and the claim has to be
# derived from the rows, not typed in beside them.
import re
from sklearn.feature_extraction.text import TfidfVectorizer

X_word = TfidfVectorizer(**D.INDEPENDENT_TFIDF_PARAMS).fit_transform(df_dedup["instruction"])
word_pairs = D.nearest_train_pairs(clean_tr, clean_te, X_word, df_dedup,
                                   threshold=D.NEAR_DUP_THRESHOLD)
identical = word_pairs[word_pairs["similarity"] >= 0.999].copy()

print(f"clean/test rows within {D.NEAR_DUP_THRESHOLD} of a same-intent training row, "
      f"in word(1,2): {len(word_pairs)} ({100*len(word_pairs)/len(clean_te):.2f}%)")
print(f"of those, word-IDENTICAL (cosine 1.0): {len(identical)}")
print()

# Each pair is identical for one of two reasons, and the difference matters:
#   - the word token pattern drops punctuation, casing and single characters,
#     so two sentences differing only in those are the same bag of tokens;
#   - min_df=2 drops any token seen once in the corpus, which on this dataset
#     means the deliberate typos - so the one word that distinguishes two
#     sentences can be pruned out of existence.
def bag_only(text):
    """What the word analyser actually keeps: lowercase alphanumeric tokens 2+ chars."""
    return " ".join(sorted(re.findall(r"\b\w\w+\b", str(text).lower())))

identical["cause"] = [
    "punctuation / casing only" if bag_only(r.eval_instruction) == bag_only(r.train_instruction)
    else "a rare token was pruned by min_df=2"
    for r in identical.itertuples()
]
causes = identical["cause"].value_counts()
print(causes.to_string())
print()

for cause in causes.index:
    example = identical[identical["cause"] == cause].iloc[0]
    print(f"  {cause}:")
    print(f"     test : {example['eval_instruction']}")
    print(f"     train: {example['train_instruction']}")

# What does the CLUSTERING space say about these same pairs?
where = D.position_index(df_dedup)
char_sims = [float((X[where[r.eval_row_id]] @ X[where[r.train_row_id]].T).toarray()[0, 0])
             for r in identical.itertuples()]

print()
print(f"char_wb similarity of those same pairs: min {min(char_sims):.3f} | "
      f"median {np.median(char_sims):.3f} | max {max(char_sims):.3f}")
print(f"every one is below the {D.NEAR_DUP_THRESHOLD} clustering threshold, which is exactly")
print("why the families were right to keep them apart.")
print()
print("So the word-space residual is a property of the MEASUREMENT SPACE, not of the")
print("split: the word analyser throws away the very characters - punctuation, casing,")
print("and rare typo tokens - that character n-grams use to tell these sentences apart.")
print("That is a far better answer to 'how do you know your split is clean' than a zero")
print("which could not have come out any other way.")
print()
print("Second-order, and it belongs in the limitations paragraph: on these rows the")
print("word-level TF-IDF baseline is not merely uncertain, it is BLIND - two sentences")
print("with different wording share one feature vector, so the model cannot tell them")
print("apart even in principle. That is part of why char_wb(3,5) scores 0.9928 against")
print("word(1,2)'s 0.9787 on this dataset.")
# The two causes and one worked example of each are printed above; that is the
# finding. The 38-row frame behind it is a dump and is not written to disk.

clean/test rows within 0.9 of a same-intent training row, in word(1,2): 38 (1.79%)
of those, word-IDENTICAL (cosine 1.0): 7

cause
a rare token was pruned by min_df=2    4
punctuation / casing only              3

  a rare token was pruned by min_df=2:
     test : I can't speak with a persno
     train: can i speak with a humanagent
  punctuation / casing only:
     test : can i order from {{Delivery City}}
     train: can I order from {{Delivery City}}?

char_wb similarity of those same pairs: min 0.275 | median 0.583 | max 0.898
every one is below the 0.9 clustering threshold, which is exactly
why the families were right to keep them apart.

So the word-space residual is a property of the MEASUREMENT SPACE, not of the
split: the word analyser throws away the very characters - punctuation, casing,
and rare typo tokens - that character n-grams use to tell these sentences apart.
That is a far better answer to 'how do you know your split is clean' than a zero
which could not have come ou

## 1.5 - Write the six CSVs, the manifest, and verify the round trip

`response` is not written. `flags` and `category` are, because error slicing and category-level metrics need them - but neither is ever a feature.

The manifest is the identity card: seed, threshold, row counts, a sha256 of the texts+labels of each split, the hash of the raw file and the library versions. The CSVs are committed beside it; the manifest is what proves that a rebuild produced the same thing. Without it, an upstream dataset update or a scikit-learn major bump moves the split and nobody notices.

In [31]:
processed = REPO_ROOT / "data" / "processed"
splits = {
    "clean": {"train": clean_tr, "val": clean_va, "test": clean_te},
    "naive": {"train": naive_tr, "val": naive_va, "test": naive_te},
}

for name, parts in splits.items():
    (processed / name).mkdir(parents=True, exist_ok=True)
    for part, frame in parts.items():
        out = frame[D.SPLIT_COLUMNS]
        out.to_csv(processed / name / f"{part}.csv", index=False, encoding="utf-8")
        print(f"wrote {name}/{part}.csv  {len(out):>6} rows  {list(out.columns)}")

wrote clean/train.csv    9893 rows  ['row_id', 'instruction', 'intent', 'category', 'flags', 'dup_group']
wrote clean/val.csv    2120 rows  ['row_id', 'instruction', 'intent', 'category', 'flags', 'dup_group']
wrote clean/test.csv    2120 rows  ['row_id', 'instruction', 'intent', 'category', 'flags', 'dup_group']
wrote naive/train.csv   17187 rows  ['row_id', 'instruction', 'intent', 'category', 'flags', 'dup_group']
wrote naive/val.csv    3683 rows  ['row_id', 'instruction', 'intent', 'category', 'flags', 'dup_group']
wrote naive/test.csv    3684 rows  ['row_id', 'instruction', 'intent', 'category', 'flags', 'dup_group']


### 1.5b - full_corpus.csv: every raw row, its family, its side

The six CSVs above are the *classification* files - one representative per family,
`response`-free, frozen. But the collapse keeps 14,133 of 24,554 rows; the other
~10.4k non-representative rows would otherwise simply vanish, and the ~2.3k
exact-duplicate rows never even reached the clustering. Letting them vanish silently is
the problem: 42% of the corpus would be gone with no record of what went, which family
it belonged to, or which side of the split that family landed on.

`full_corpus.csv` is the registry that keeps them: **every** non-empty raw row,
stamped with its family id, the split side its family's representative landed on
(the *clean* split - the naive control keeps its own six CSVs and plays no role
here), and two role flags. Nothing trains on this file. Its consumers are
`src.data.train_side_rows()`, which rebuilds the uncollapsed training set from it for
the family-collapse ablation, and anyone auditing the split.

Three properties, all asserted below:

1. **Coverage** - every raw row has exactly one side; exact duplicates inherit their
   twin's family (unambiguous: the dataset has zero exact label conflicts).
2. **Family unity** - no family spans two sides, so ANY view of this file filtered by
   `split` is leak-free: it cannot contain a sibling of a sentence held out on the
   other side. Contamination is impossible by construction, not by convention.
3. **Consistency** - filtering to `is_representative` reproduces the six committed
   CSVs row for row.

`response` is deliberately absent from this file too, for the same reason as the six
CSVs: nothing under `data/processed/` ever contains the answer text, so trap 1 stays
impossible rather than merely discouraged. Anything that needs the response text joins
it back from `data/raw/` via `row_id`.


In [32]:
full_corpus = D.build_full_corpus(
    df_ws, df_dedup, {"train": clean_tr, "val": clean_va, "test": clean_te})
full_corpus.to_csv(processed / "full_corpus.csv", index=False, encoding="utf-8")

print(f"wrote full_corpus.csv  {len(full_corpus):,} rows  {list(full_corpus.columns)}")
print("\nrows per side (whole families, all members):")
print(full_corpus["split"].value_counts().to_string())
print(f"\nrepresentatives: {full_corpus['is_representative'].sum():,} (= exactly the rows of the three clean CSVs)"
      f"   exact duplicates re-attached: {full_corpus['is_exact_duplicate'].sum():,}")


wrote full_corpus.csv  26,872 rows  ['row_id', 'instruction', 'intent', 'category', 'flags', 'dup_group', 'split', 'is_representative', 'is_exact_duplicate']

rows per side (whole families, all members):
split
train    18523
val       4332
test      4017

representatives: 14,133 (= exactly the rows of the three clean CSVs)   exact duplicates re-attached: 2,318


In [33]:
# Property 1 - coverage: every raw row is present and has a side
assert len(full_corpus) == len(df_ws) and full_corpus["split"].notna().all()
# Property 2 - family unity: no family spans two sides
assert (full_corpus.groupby("dup_group")["split"].nunique() == 1).all()
# Property 3 - the representative view IS the committed clean split
for part, frame in [("train", clean_tr), ("val", clean_va), ("test", clean_te)]:
    view = full_corpus[full_corpus["is_representative"] & (full_corpus["split"] == part)]
    assert set(view["row_id"]) == set(frame["row_id"]), f"clean/{part} mismatch"
print("full_corpus verified: every raw row has one side, no family spans two sides,")
print("and the representative view reproduces the clean split exactly.")


full_corpus verified: every raw row has one side, no family spans two sides,
and the representative view reproduces the clean split exactly.


In [34]:
counts = {
    "raw": len(df_raw),
    "after_whitespace": len(df_ws),
    "after_exact_duplicates": len(df_dedup),
    "n_families": int(n_groups),
    "clean_total": len(reps_random),
}
manifest = D.build_manifest(raw_path, counts, splits,
                            library_versions=LIBRARY_VERSIONS,
                            full_corpus=full_corpus)
manifest["leakage"] = {"naive_before_cleaning": leak_before,
                       "naive_committed": leak_naive,
                       "clean_committed": leak_clean}
D.write_json(manifest, processed / "split_manifest.json")
print(json.dumps(manifest, indent=2)[:1200], "\n...")

{
  "seed": 42,
  "near_dup_threshold": 0.9,
  "scheme": "70/15/15 stratified on intent, two-step",
  "source": {
    "hf_repo": "bitext/Bitext-customer-support-llm-chatbot-training-dataset",
    "filename": "Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv",
    "sha256": "6f81102b0100b97b8468eb04368033a23206bf1fde9d53500d5806ec1001a434"
  },
  "tfidf_params": {
    "analyzer": "char_wb",
    "ngram_range": [
      3,
      5
    ],
    "min_df": 2,
    "sublinear_tf": true
  },
  "counts": {
    "raw": 26872,
    "after_whitespace": 26872,
    "after_exact_duplicates": 24554,
    "n_families": 14133,
    "clean_total": 14133
  },
  "splits": {
    "clean": {
      "train": {
        "n_rows": 9893,
        "sha256": "6d674bcb7d8fd254705cf847ebeefae9e3f9e356f385bb3c5b365819de78d177"
      },
      "val": {
        "n_rows": 2120,
        "sha256": "cee4cf6f0118f9a8fa35946fc91babebd25a6273108dfea1c473ada89c053d81"
      },
      "test": {
        "n_rows": 2120,
  

In [35]:
# The round trip: read the six files back off disk, re-hash them, and compare
# to the manifest. This is the assert the bootstrap cell of every other
# notebook will run - a reproducibility claim that checks itself.
reloaded = {
    name: {part: pd.read_csv(processed / name / f"{part}.csv")
           for part in ("train", "val", "test")}
    for name in ("clean", "naive")
}
reloaded_full = pd.read_csv(processed / "full_corpus.csv")
D.verify_against_manifest(manifest, reloaded, full_corpus=reloaded_full)
print("round trip OK - the files on disk match split_manifest.json exactly,")
print("full_corpus.csv included.")

round trip OK - the files on disk match split_manifest.json exactly,
full_corpus.csv included.


## 1.6 - The two artifacts

Two small JSON files, created once, committed, and never moved. They are the contract every later step leans on.

| file | contents | why it must stay stable |
|---|---|---|
| `labels.json` | the 27 intents, sorted | the model knows integers 0-26, not names. If the order shifts between runs, a previously saved checkpoint returns wrong answers with **no error** |
| `intent2cat.json` | intent -> category | built from the file, not the docs. The 11 categories are looked up, never predicted |

`labels.json` is the single deliberate exception to "fit on train only": class order is a contract, not a learned parameter, so it is built from all 27 labels on purpose.

In [36]:
artifacts = REPO_ROOT / "artifacts"
labels = D.build_labels(df_raw)
intent2cat = D.build_intent2cat(df_raw)

assert len(labels) == 27 and labels == sorted(labels)
assert set(intent2cat) == set(labels) and len(set(intent2cat.values())) == 11

D.write_json(labels, artifacts / "labels.json")
D.write_json(intent2cat, artifacts / "intent2cat.json")

print("labels.json        :", labels[:3], "...", labels[-2:])
print("intent2cat.json    :", dict(list(intent2cat.items())[:3]), "...")


labels.json        : ['cancel_order', 'change_order', 'change_shipping_address'] ... ['track_order', 'track_refund']
intent2cat.json    : {'cancel_order': 'ORDER', 'change_order': 'ORDER', 'change_shipping_address': 'SHIPPING'} ...


## 1.7 - Three summaries, printed

Everything this notebook still owes the report is a number, so the numbers are
printed here and no figure files are produced:

1. **Class distribution before and after cleaning** - answers "describe the data", and the "after" is what later justifies reporting macro-F1.
2. **Text length in tokens** with the p99 - this sets `max_length` for training, and training time grows roughly with the square of sequence length.
3. **Flag frequency** - the generation mechanism of the dataset in one table, and the setup for error slicing in stage 5.

The test <-> train similarity distribution needs nothing further here: 1.5 already
reports it as percentages for both splits.


In [ ]:
# Class distribution before and after cleaning
before = df_raw["intent"].value_counts()
after = reps_random["intent"].value_counts()

print(f"raw   : {before.min()}-{before.max()} rows per intent (ratio {before.max()/before.min():.2f})")
print(f"clean : {after.min()}-{after.max()} rows per intent (ratio {after.max()/after.min():.2f})")
print("\nThe cleaned dataset is markedly less balanced than the raw one:")
print("the raw dataset is balanced (1.05:1), the cleaned dataset is NOT (4.06:1).")
print("Intents whose templates were expanded most aggressively - cancel_order keeps")
print("20% of its rows, place_order keeps 82% - lose the most. So macro-F1 is not a")
print("stylistic preference here: after cleaning, accuracy really can hide a small")
print("class the model never gets right.")

In [ ]:
# Text length in TOKENS, and the p99 that sets max_length
import os, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
warnings.filterwarnings("ignore")

lengths = D.token_lengths(df_dedup["instruction"])
p99 = int(np.percentile(lengths, 99))
max_length = int(2 ** np.ceil(np.log2(p99)))

print(f"mean {lengths.mean():.1f} | median {int(np.median(lengths))} | p95 {int(np.percentile(lengths,95))} "
      f"| p99 {p99} | max {lengths.max()}")
print(f"chosen max_length = {max_length} (next power of two above p99)")
print(f"rows truncated at max_length={max_length}: {(lengths > max_length).sum()} "
      f"({100*(lengths > max_length).mean():.3f}%)")
print(f"the naive default of 512 would cost roughly (512/{max_length})^2 = "
      f"{(512/max_length)**2:.0f}x the attention compute for the same result.")


In [ ]:
# Flag frequency after de-duplication
counts_f = Counter("".join(df_dedup["flags"].astype(str)))
names = {"B": "basic structure", "L": "synonym/semantic", "Q": "colloquial", "I": "question form",
         "Z": "typos", "M": "morphology", "C": "coordination", "K": "keyword-only",
         "E": "abbreviation", "P": "politeness", "W": "offensive", "N": "negation",
         "S": "undocumented", "V": "undocumented"}

flags_dedup = pd.DataFrame(
    [{"flag": f, "meaning": names.get(f, "?"), "rows": n,
      "pct_of_dedup": round(100 * n / len(df_dedup), 2)}
     for f, n in counts_f.most_common()])
display(flags_dedup)

print("flags per row - mean:", round(df_dedup['flags'].str.len().mean(), 2))


## 1.8 - Pre-registered prediction (written before anything is trained)

Recorded now, on purpose, so that later analysis compares the confusion pairs against a prediction made **before** seeing any result. A prediction that holds is the most convincing paragraph in the analysis chapter; one that is refuted is more interesting still.

Expected confusion pairs:

- `get_invoice` <-> `check_invoice` (one verb apart)
- `get_refund` <-> `track_refund` <-> `check_refund_policy`
- `cancel_order` <-> `change_order` <-> `check_cancellation_fee`
- `contact_customer_service` <-> `contact_human_agent` (the most delicate pair in the file)
- `change_shipping_address` <-> `set_up_shipping_address`
- `complaint` <-> `payment_issue`

Expected robustness ranking from the flag slices: `Z` (typos) and `K` (keyword-only) hardest; `P` (politeness) no measurable effect.

## Summary

Everything below is measured, not assumed. Nothing here required a GPU.

In [ ]:
summary = [
    f"DATA             raw {len(df_raw):,} rows / 27 intents / 11 categories  (verified)",
    f"CLEANING         -> {len(df_dedup):,} after exact duplicates ({len(df_ws)-len(df_dedup):,} removed)",
    f"                 -> {len(reps_random):,} after family clustering at {D.NEAR_DUP_THRESHOLD}",
    f"                    ({n_groups:,} families, mean size {len(df_dedup)/n_groups:.2f})",
    f"LABEL CONFLICTS  {len(conflicts)} exact, but {len(twins)} rows "
    f"({twin_pct:.2f}%) sit within {D.NEAR_DUP_THRESHOLD} of a DIFFERENT intent",
    f"                 -> taxonomy granularity, NOT an accuracy ceiling "
    f"(tested in nb02: the baseline gets all such val rows right)",
    f"LEAKAGE naive    {leak_before['exact_pct']}% verbatim | {leak_before['ge_0.95_pct']}% >=0.95 | {leak_before['ge_0.90_pct']}% >=0.90",
    f"LEAKAGE clean    {leak_clean['exact_pct']}% verbatim | {leak_clean['ge_0.95_pct']}% >=0.95 | {leak_clean['ge_0.90_pct']}% >=0.90",
    f"                 same-intent >=0.90 is 0.00%, but that is ZERO BY CONSTRUCTION;",
    f"                 in the independent word(1,2) space it is {word_residual}%",
    f"CLEANING AUDIT   whitespace normalisation merges "
    f"{ws_impact['extra_pairs_merged']} extra duplicate pairs",
    f"SPLIT            clean {len(clean_tr)}/{len(clean_va)}/{len(clean_te)}   naive {len(naive_tr)}/{len(naive_va)}/{len(naive_te)}",
    f"                 seed {D.SPLIT_SEED}, stratified on intent, all 27 present in all six files",
    f"FULL CORPUS      {len(full_corpus):,} rows registered - family + split side + role for every raw row",
    f"MAX_LENGTH       p99 = {p99} tokens -> max_length = {max_length}",
    "ARTIFACTS        labels.json, intent2cat.json (all asserts passed)",
]
print("\n".join(summary))